In [4]:
import cv2
import mediapipe as mp
import numpy as np
import joblib

# Load smile model
model = joblib.load("../models/smile_model.pkl")

# Load filters
glasses = cv2.imread('../filters/glasses.png', cv2.IMREAD_UNCHANGED)
fire = cv2.imread('../filters/fire.png', cv2.IMREAD_UNCHANGED)

# MediaPipe face mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False)

# Utility to overlay PNG with alpha channel
def overlay_image_alpha(img, img_overlay, x, y):
    h, w = img_overlay.shape[:2]
    alpha = img_overlay[:, :, 3] / 255.0
    for c in range(3):
        if y + h <= img.shape[0] and x + w <= img.shape[1] and y >= 0 and x >= 0:
            img[y:y+h, x:x+w, c] = (
                img_overlay[:, :, c] * alpha +
                img[y:y+h, x:x+w, c] * (1 - alpha) 
            )
    return img

# Webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = face_mesh.process(frame_rgb)

    if result.multi_face_landmarks:
        for face_landmarks in result.multi_face_landmarks:
            ih, iw, _ = frame.shape

            # Landmarks for eyes and mouth
            left_eye = face_landmarks.landmark[33]
            right_eye = face_landmarks.landmark[263]
            top_lip = face_landmarks.landmark[13]
            bottom_lip = face_landmarks.landmark[14]

            # Pixel positions
            x1_eye = int(left_eye.x * iw)
            x2_eye = int(right_eye.x * iw)
            eye_y = int(left_eye.y * ih)
            y_top = int(top_lip.y * ih)
            y_bottom = int(bottom_lip.y * ih)

            # Always display enlarged glasses
            eye_distance = x2_eye - x1_eye
            scale_factor = 2.0  # Bigger specs    
            glasses_width = int(eye_distance * scale_factor)   
            x_offset_glasses = x1_eye - (glasses_width - eye_distance) // 2
            y_offset_glasses = eye_y - 60# Moved specs further upward (this makes glasses move upward by increasing the nnumber)
            glasses_resized = cv2.resize(
                glasses,
                (glasses_width, int(glasses_width * glasses.shape[0] / glasses.shape[1]))
            )
            frame = overlay_image_alpha(frame, glasses_resized, x_offset_glasses, y_offset_glasses)

            # Compute mouth height for smile detection 
            mouth_height = abs(y_bottom - y_top)
            prediction = model.predict([[mouth_height]])[0]

            # If smiling, show fire
            if prediction == 1:
                fire_width = eye_distance
                fire_resized = cv2.resize(
                    fire,
                    (fire_width, int(fire_width * fire.shape[0] / fire.shape[1]))
                )
                left_corner = face_landmarks.landmark[61]
                right_corner = face_landmarks.landmark[291]
                center_mouth_x = int((left_corner.x + right_corner.x) / 2 * iw)
                fire_x_offset = center_mouth_x - fire_width // 2 + 45
                fire_y_offset = int((y_bottom + y_top) / 2)
                frame = overlay_image_alpha(frame, fire_resized, fire_x_offset, fire_y_offset)
                print(mouth_height)

            status = "Smiling" if prediction == 1 else "Not Smiling"
            cv2.putText(frame, status, (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Snapchat Filter with ML", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\valid

12
16
18
19
18


c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\valid

18
16
17
18
19


c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\valid

...

15
15
15
15
15


c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\valid

14
14
14
14
14


c:\Users\mailt\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


15
